In [1]:
import pandas as pd
import numpy as np

from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_csv(
    "data/processed/content_candidates.csv"
)

available_genres = [
    "Action", "Adventure", "Animation",
    "Children", "Comedy", "Crime",
    "Documentary", "Drama", "Fantasy",
    "Film-Noir", "Horror", "Musical",
    "Mystery", "Romance", "Sci-Fi",
    "Thriller", "War", "Western"
]

In [2]:
selected_genres = [
    "Action",
    "Sci-Fi",
    "Thriller"
]

user_profile = pd.Series(
    0.0,
    index=available_genres
)

user_profile[selected_genres] = 1.0

print("=== Initial User Profile ===")
print(user_profile)

=== Initial User Profile ===
Action         1.0
Adventure      0.0
Animation      0.0
Children       0.0
Comedy         0.0
Crime          0.0
Documentary    0.0
Drama          0.0
Fantasy        0.0
Film-Noir      0.0
Horror         0.0
Musical        0.0
Mystery        0.0
Romance        0.0
Sci-Fi         1.0
Thriller       1.0
War            0.0
Western        0.0
dtype: float64


In [3]:
target_movie = movies[
    movies["title"] == "Matrix, The (1999)"
]

print("=== Selected Movie ===")

print(
    target_movie[
        ["movieId", "title", "genres"]
    ]
)

=== Selected Movie ===
      movieId               title                  genres
1939     2571  Matrix, The (1999)  Action|Sci-Fi|Thriller


In [4]:
movie_genre_vector = target_movie[
    available_genres
].iloc[0]

print("=== Movie Genre Vector ===")
print(movie_genre_vector)

=== Movie Genre Vector ===
Action         1
Adventure      0
Animation      0
Children       0
Comedy         0
Crime          0
Documentary    0
Drama          0
Fantasy        0
Film-Noir      0
Horror         0
Musical        0
Mystery        0
Romance        0
Sci-Fi         1
Thriller       1
War            0
Western        0
Name: 1939, dtype: int64


In [5]:
new_rating = 5.0
baseline_rating = 3.5

preference_weight = new_rating - baseline_rating

print("=== Rating Feedback ===")
print("New rating:", new_rating)
print("Preference weight:", preference_weight)

=== Rating Feedback ===
New rating: 5.0
Preference weight: 1.5


In [6]:
number_of_genres = movie_genre_vector.sum()

print("Number of movie genres:", number_of_genres)

Number of movie genres: 3


In [7]:
genre_weight = preference_weight / number_of_genres

print("Genre weight:", genre_weight)

Genre weight: 0.5


In [8]:
updated_profile = user_profile.copy()

updated_profile += (
    movie_genre_vector * genre_weight
)

print("=== Updated User Profile ===")
print(updated_profile)

=== Updated User Profile ===
Action         1.5
Adventure      0.0
Animation      0.0
Children       0.0
Comedy         0.0
Crime          0.0
Documentary    0.0
Drama          0.0
Fantasy        0.0
Film-Noir      0.0
Horror         0.0
Musical        0.0
Mystery        0.0
Romance        0.0
Sci-Fi         1.5
Thriller       1.5
War            0.0
Western        0.0
dtype: float64


In [9]:
profile_comparison = pd.DataFrame({
    "Before": user_profile,
    "After": updated_profile
})

profile_comparison["Change"] = (
    profile_comparison["After"]
    - profile_comparison["Before"]
)

print("=== User Profile Comparison ===")
print(profile_comparison)

=== User Profile Comparison ===
             Before  After  Change
Action          1.0    1.5     0.5
Adventure       0.0    0.0     0.0
Animation       0.0    0.0     0.0
Children        0.0    0.0     0.0
Comedy          0.0    0.0     0.0
Crime           0.0    0.0     0.0
Documentary     0.0    0.0     0.0
Drama           0.0    0.0     0.0
Fantasy         0.0    0.0     0.0
Film-Noir       0.0    0.0     0.0
Horror          0.0    0.0     0.0
Musical         0.0    0.0     0.0
Mystery         0.0    0.0     0.0
Romance         0.0    0.0     0.0
Sci-Fi          1.0    1.5     0.5
Thriller        1.0    1.5     0.5
War             0.0    0.0     0.0
Western         0.0    0.0     0.0


In [10]:
def update_user_profile(
    user_profile,
    movie_id,
    rating,
    movies,
    available_genres,
    baseline=3.5
):
    # 1. Validate the rating range
    if rating < 0.5 or rating > 5.0:
        raise ValueError("Rating must be between 0.5 and 5.0")

    if rating * 2 != round(rating * 2):
        raise ValueError("Rating must be in 0.5 increments")

    # 2. Search for the movie
    movie = movies[movies["movieId"] == movie_id]

    if movie.empty:
        raise ValueError("Movie not found")

    # 3. Retrieve the genre vector
    genre_vector = movie[available_genres].iloc[0]

    # 4. Count the movie's genres
    genre_count = genre_vector.sum()

    if genre_count == 0:
        return user_profile.copy()

    # 5. Calculate preference change based on the rating
    preference_weight = rating - baseline
    genre_weight = preference_weight / genre_count

    # 6. Update the user profile
    new_profile = user_profile.copy()

    new_profile += genre_vector * genre_weight

    return new_profile

In [11]:
initial_profile = pd.Series(
    0.0,
    index=available_genres
)

initial_profile[
    ["Action", "Sci-Fi", "Thriller"]
] = 1.0

print("=== Initial Profile ===")
print(initial_profile)

=== Initial Profile ===
Action         1.0
Adventure      0.0
Animation      0.0
Children       0.0
Comedy         0.0
Crime          0.0
Documentary    0.0
Drama          0.0
Fantasy        0.0
Film-Noir      0.0
Horror         0.0
Musical        0.0
Mystery        0.0
Romance        0.0
Sci-Fi         1.0
Thriller       1.0
War            0.0
Western        0.0
dtype: float64


In [12]:
profile_after_matrix = update_user_profile(
    user_profile=initial_profile,
    movie_id=2571,
    rating=5.0,
    movies=movies,
    available_genres=available_genres
)

print("=== Profile After Rating The Matrix ===")
print(profile_after_matrix)

=== Profile After Rating The Matrix ===
Action         1.5
Adventure      0.0
Animation      0.0
Children       0.0
Comedy         0.0
Crime          0.0
Documentary    0.0
Drama          0.0
Fantasy        0.0
Film-Noir      0.0
Horror         0.0
Musical        0.0
Mystery        0.0
Romance        0.0
Sci-Fi         1.5
Thriller       1.5
War            0.0
Western        0.0
dtype: float64


In [13]:
profile_after_toy_story = update_user_profile(
    user_profile=profile_after_matrix,
    movie_id=1,
    rating=5.0,
    movies=movies,
    available_genres=available_genres
)

print("=== Profile After Rating Toy Story ===")
print(profile_after_toy_story)

=== Profile After Rating Toy Story ===
Action         1.5
Adventure      0.3
Animation      0.3
Children       0.3
Comedy         0.3
Crime          0.0
Documentary    0.0
Drama          0.0
Fantasy        0.3
Film-Noir      0.0
Horror         0.0
Musical        0.0
Mystery        0.0
Romance        0.0
Sci-Fi         1.5
Thriller       1.5
War            0.0
Western        0.0
dtype: float64


In [14]:
def recommend_movies(
    user_profile,
    movies,
    available_genres,
    watched_movie_ids=None,
    alpha=0.7,
    top_n=10
):
    # 1. Get the list of rated movies
    if watched_movie_ids is None:
        watched_movie_ids = []

    # 2. Prepare the movie genre matrix
    movie_vectors = movies[available_genres].to_numpy()

    # 3. Prepare the user vector
    user_vector = user_profile[
        available_genres
    ].to_numpy().reshape(1, -1)

    # 4. Calculate cosine similarity
    similarity_scores = cosine_similarity(
        user_vector,
        movie_vectors
    )[0]

    # 5. Store results
    results = movies.copy()

    results["genre_similarity"] = similarity_scores

    # 6. Calculate final recommendation scores
    results["recommendation_score"] = (
        alpha * results["genre_similarity"]
        + (1 - alpha) * results["normalized_rating"]
    )

    # 7. Exclude watched movies
    results = results[
        ~results["movieId"].isin(watched_movie_ids)
    ]

    # 8. Exclude movies with zero genre similarity
    results = results[
        results["genre_similarity"] > 0
    ]

    # 9. Sort by recommendation score
    results = results.sort_values(
        by="recommendation_score",
        ascending=False
    )

    return results.head(top_n)

In [15]:
initial_recommendations = recommend_movies(
    user_profile=initial_profile,
    movies=movies,
    available_genres=available_genres
)

print("=== Initial Recommendations ===")

print(
    initial_recommendations[
        [
            "title",
            "genres",
            "genre_similarity",
            "recommendation_score"
        ]
    ].to_string(index=False)
)

=== Initial Recommendations ===
                            title                      genres  genre_similarity  recommendation_score
               Matrix, The (1999)      Action|Sci-Fi|Thriller               1.0              0.967471
              Blade Runner (1982)      Action|Sci-Fi|Thriller               1.0              0.951729
           Terminator, The (1984)      Action|Sci-Fi|Thriller               1.0              0.925507
            V for Vendetta (2006) Action|Sci-Fi|Thriller|IMAX               1.0              0.922879
               Equilibrium (2002)      Action|Sci-Fi|Thriller               1.0              0.916673
                  eXistenZ (1999)      Action|Sci-Fi|Thriller               1.0              0.908918
                 Hangar 18 (1980)      Action|Sci-Fi|Thriller               1.0              0.897358
                 They Live (1988)      Action|Sci-Fi|Thriller               1.0              0.888689
Captain America: Civil War (2016)      Action|Sci-

In [16]:
watched_movie_ids = [2571, 1]

updated_recommendations = recommend_movies(
    user_profile=profile_after_toy_story,
    movies=movies,
    available_genres=available_genres,
    watched_movie_ids=watched_movie_ids
)

print("=== Updated Recommendations ===")

print(
    updated_recommendations[
        [
            "title",
            "genres",
            "genre_similarity",
            "recommendation_score"
        ]
    ].to_string(index=False)
)

=== Updated Recommendations ===
                               title                           genres  genre_similarity  recommendation_score
                 Blade Runner (1982)           Action|Sci-Fi|Thriller          0.968246              0.929501
              Terminator, The (1984)           Action|Sci-Fi|Thriller          0.968246              0.903279
               V for Vendetta (2006)      Action|Sci-Fi|Thriller|IMAX          0.968246              0.900651
                  Equilibrium (2002)           Action|Sci-Fi|Thriller          0.968246              0.894445
                     eXistenZ (1999)           Action|Sci-Fi|Thriller          0.968246              0.886690
                    Hangar 18 (1980)           Action|Sci-Fi|Thriller          0.968246              0.875130
                    They Live (1988)           Action|Sci-Fi|Thriller          0.968246              0.866461
   Captain America: Civil War (2016)           Action|Sci-Fi|Thriller          0.968246 

In [17]:
rating_history = {
    2571: 5.0,
    1: 5.0
}

print("=== Rating History ===")
print(rating_history)

=== Rating History ===
{2571: 5.0, 1: 5.0}


In [18]:
rating_history[2571] = 4.0

print(rating_history)

{2571: 4.0, 1: 5.0}


In [19]:
def build_user_profile(
    selected_genres,
    rating_history,
    movies,
    available_genres,
    baseline=3.5
):
    # 1. Create the initial user profile
    profile = pd.Series(
        0.0,
        index=available_genres
    )

    profile[selected_genres] = 1.0

    # 2. Apply rating records one at a time
    for movie_id, rating in rating_history.items():

        profile = update_user_profile(
            user_profile=profile,
            movie_id=movie_id,
            rating=rating,
            movies=movies,
            available_genres=available_genres,
            baseline=baseline
        )

    return profile

In [20]:
rating_history = {
    2571: 5.0,
    1: 5.0
}

profile_before_edit = build_user_profile(
    selected_genres=selected_genres,
    rating_history=rating_history,
    movies=movies,
    available_genres=available_genres
)

print("=== Profile Before Rating Edit ===")
print(profile_before_edit)

=== Profile Before Rating Edit ===
Action         1.5
Adventure      0.3
Animation      0.3
Children       0.3
Comedy         0.3
Crime          0.0
Documentary    0.0
Drama          0.0
Fantasy        0.3
Film-Noir      0.0
Horror         0.0
Musical        0.0
Mystery        0.0
Romance        0.0
Sci-Fi         1.5
Thriller       1.5
War            0.0
Western        0.0
dtype: float64


In [21]:
rating_history[2571] = 4.0

profile_after_edit = build_user_profile(
    selected_genres=selected_genres,
    rating_history=rating_history,
    movies=movies,
    available_genres=available_genres
)

print("=== Profile After Rating Edit ===")
print(profile_after_edit)

=== Profile After Rating Edit ===
Action         1.166667
Adventure      0.300000
Animation      0.300000
Children       0.300000
Comedy         0.300000
Crime          0.000000
Documentary    0.000000
Drama          0.000000
Fantasy        0.300000
Film-Noir      0.000000
Horror         0.000000
Musical        0.000000
Mystery        0.000000
Romance        0.000000
Sci-Fi         1.166667
Thriller       1.166667
War            0.000000
Western        0.000000
dtype: float64


In [22]:
profile_comparison = pd.DataFrame({
    "Before": profile_before_edit,
    "After": profile_after_edit
})

profile_comparison["Change"] = (
    profile_comparison["After"]
    - profile_comparison["Before"]
)

print("=== Profile Comparison ===")
print(profile_comparison)

=== Profile Comparison ===
             Before     After    Change
Action          1.5  1.166667 -0.333333
Adventure       0.3  0.300000  0.000000
Animation       0.3  0.300000  0.000000
Children        0.3  0.300000  0.000000
Comedy          0.3  0.300000  0.000000
Crime           0.0  0.000000  0.000000
Documentary     0.0  0.000000  0.000000
Drama           0.0  0.000000  0.000000
Fantasy         0.3  0.300000  0.000000
Film-Noir       0.0  0.000000  0.000000
Horror          0.0  0.000000  0.000000
Musical         0.0  0.000000  0.000000
Mystery         0.0  0.000000  0.000000
Romance         0.0  0.000000  0.000000
Sci-Fi          1.5  1.166667 -0.333333
Thriller        1.5  1.166667 -0.333333
War             0.0  0.000000  0.000000
Western         0.0  0.000000  0.000000


In [23]:
watched_movie_ids = list(rating_history.keys())

recommendations_after_edit = recommend_movies(
    user_profile=profile_after_edit,
    movies=movies,
    available_genres=available_genres,
    watched_movie_ids=watched_movie_ids
)

print("=== Recommendations After Rating Edit ===")

print(
    recommendations_after_edit[
        [
            "title",
            "genres",
            "genre_similarity",
            "recommendation_score"
        ]
    ].to_string(index=False)
)

=== Recommendations After Rating Edit ===
                               title                           genres  genre_similarity  recommendation_score
                 Blade Runner (1982)           Action|Sci-Fi|Thriller          0.949071              0.916079
              Terminator, The (1984)           Action|Sci-Fi|Thriller          0.949071              0.889857
               V for Vendetta (2006)      Action|Sci-Fi|Thriller|IMAX          0.949071              0.887228
                  Equilibrium (2002)           Action|Sci-Fi|Thriller          0.949071              0.881023
                     eXistenZ (1999)           Action|Sci-Fi|Thriller          0.949071              0.873268
                    Hangar 18 (1980)           Action|Sci-Fi|Thriller          0.949071              0.861707
Road Warrior, The (Mad Max 2) (1981) Action|Adventure|Sci-Fi|Thriller          0.892370              0.858768
                    They Live (1988)           Action|Sci-Fi|Thriller         